In [1]:
import os
os.environ.setdefault("PYTENSOR_FLAGS", "cxx=")
import numpy as np
import pymc as pm
import arviz as az
import seaborn as sns
import matplotlib.pyplot as plt

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`


In [ ]:
sns.set_theme(style="whitegrid")

In [ ]:
conversions_A = 30
total_A = 100

conversions_B = 40
total_B = 100

In [ ]:
with pm.Model() as ab_model:

    p_A = pm.Beta("p_A", alpha=1, beta=1)

    p_B = pm.Beta("p_B", alpha=1, beta=1)

    obs_A = pm.Binomial(
        "obs_A",
        n=total_A,
        p=p_A,
        observed=conversions_A
    )

    obs_B = pm.Binomial(
        "obs_B",
        n=total_B,
        p=p_B,
        observed=conversions_B
    )

    delta = pm.Deterministic(
        "delta",
        p_B - p_A
    )

    trace_ab = pm.sample(
        draws=200,
        tune=200,
        chains=2,
        cores=1,
        progressbar=False,
        compile_kwargs={"mode": "NUMBA"},
        target_accept=0.95,
        random_seed=42
    )

In [ ]:
az.summary(
    trace_ab,
    var_names=["p_A", "p_B", "delta"]
)

In [ ]:
az.plot_trace(
    trace_ab,
    var_names=["p_A", "p_B", "delta"],
    figsize=(14, 8)
)

plt.tight_layout();

In [ ]:
az.plot_posterior(
    trace_ab,
    var_names=["p_A", "p_B", "delta"],
    figsize=(14, 4)
)

plt.tight_layout();

In [ ]:
delta_samples = (
    trace_ab.posterior["delta"]
    .stack(sample=("chain", "draw"))
    .values
)

prob_B_better = np.mean(delta_samples > 0)

print(f"P(B > A) = {prob_B_better:.3f}")

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    delta_samples,
    bins=50,
    kde=True
)

plt.axvline(
    0,
    color="red",
    linestyle="--"
)

plt.xlabel("p_B - p_A")

plt.title("Posterior distribution of difference");